# Week 35, Day 5: Decorator (the pattern)

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w035d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Prices:
    def get(self, item):
        return {"rice": 450, "oil": 900}.get(item)

class Logged:
    def __init__(self, inner):
        self.inner = inner
        self.log = []
    def get(self, item):
        self.log.append(item)
        return self.inner.get(item)

class WithDefault:
    def __init__(self, inner, default):
        self.inner = inner
        self.default = default
    def get(self, item):
        value = self.inner.get(item)
        return self.default if value is None else value

logged = Logged(Prices())
source = WithDefault(logged, 0)
print(source.get("rice"), source.get("yam"), logged.log)

In [ ]:
class Database:
    def __init__(self, data):
        self.data = data
        self.queries = 0
    def get(self, key):
        self.queries += 1
        return self.data.get(key)

class Cached:
    def __init__(self, inner):
        self.inner = inner
        self.saved = {}
    def get(self, key):
        if key not in self.saved:
            self.saved[key] = self.inner.get(key)
        return self.saved[key]

class Logged:
    def __init__(self, inner, log):
        self.inner = inner
        self.log = log
    def get(self, key):
        value = self.inner.get(key)
        self.log.append(f"get({key!r}) -> {value!r}")
        return value

class Upper:
    def __init__(self, inner):
        self.inner = inner
    def get(self, key):
        value = self.inner.get(key)
        return value.upper() if isinstance(value, str) else value

db = Database({"city": "Lagos", "country": "Nigeria"})
log = []
source = Upper(Cached(Logged(db, log)))
print([source.get(k) for k in ["city", "city", "country", "city"]])
print(db.queries, log)

In [ ]:
class Database:
    def __init__(self, data):
        self.data = data
    def get(self, key):
        return self.data.get(key)

class Cached:
    def __init__(self, inner):
        self.inner, self.saved = inner, {}
    def get(self, key):
        if key not in self.saved:
            self.saved[key] = self.inner.get(key)
        return self.saved[key]

class Logged:
    def __init__(self, inner, log):
        self.inner, self.log = inner, log
    def get(self, key):
        self.log.append(key)
        return self.inner.get(key)

inside, outside = [], []
a = Cached(Logged(Database({"k": 1}), inside))      # logs only real lookups
b = Logged(Cached(Database({"k": 1})), outside)     # logs every request
for _ in range(3):
    a.get("k")
    b.get("k")
print(inside, outside)

In [ ]:
import functools

def logged(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"{func.__name__}{args} -> {result}")
        return result
    return wrapper

def vat(pence, rate=20):
    """VAT on a price in pence."""
    return pence * rate // 100

vat = logged(vat)
vat(1000)
vat(999, rate=5)
print(vat.__name__, vat.__doc__)

In [ ]:
class Retry:
    def __init__(self, func, times=3):
        self.func = func
        self.times = times
        self.attempts = 0

    def __call__(self, *args, **kwargs):
        for attempt in range(1, self.times + 1):
            self.attempts += 1
            try:
                return self.func(*args, **kwargs)
            except ConnectionError:
                if attempt == self.times:
                    raise

failures = iter([True, True, False])
def flaky_fetch(url):
    if next(failures):
        raise ConnectionError("timed out")
    return f"<page from {url}>"

fetch = Retry(flaky_fetch, times=3)
print(fetch("example.org"), fetch.attempts)

> **Key idea.** Decorator (Gang of Four): add responsibilities to an object by wrapping it in another with the same interface, which adds behaviour before or after passing calls on; behaviours are written once and combined freely, avoiding a subclass per combination. Stacked decorators run outside-in on the way in, so their order matters. Function wrappers take a function and return `wrapper(*args, **kwargs)`, which calls the original, adds its job and returns the result; `functools.wraps` keeps the original's name and docstring. `f = deco(f)` is what the `@deco` syntax does (Phase 5). Use a callable class for decorators with settings. Adapters change interfaces; proxies keep them but control access.

### Exercise: Counting lookups

Write two decorators for anything with `get(key)`: `Counted(inner)`, which counts how many times `get` was called in an attribute `calls`, and `Cached(inner)`, which calls the inner `get` at most once per key (remembering `None` results too) and counts in `hits` the calls it answered from memory. Both pass the result through unchanged.

In [ ]:
class Source:
    def __init__(self, data):
        self.data = data

    def get(self, key):
        return self.data.get(key)

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_decorators():\n    \"counting and caching\"\n    S, Co, Ca = ns[\"Source\"], ns[\"Counted\"], ns[\"Cached\"]\n    inner = Co(S({\"a\": 1}))\n    c = Ca(inner)\n    assert [c.get(k) for k in [\"a\", \"a\", \"b\", \"b\", \"a\"]] == [1, 1, None, None, 1]\n    assert inner.calls == 2 and c.hits == 3, f\"calls {inner.calls}, hits {c.hits}\"\n    outer = Co(Ca(S({\"a\": 1})))\n    for _ in range(4):\n        outer.get(\"a\")\n    assert outer.calls == 4")

### Exercise: Function wrappers

Write two function wrappers, each using `functools.wraps`:

- `count_calls(func)`: returns a wrapper with an attribute `calls`, counting how many times it's been called;
- `recorded(func, log)`: returns a wrapper that appends lines like `vat(1000, rate=5) -> 50` to the list `log` (positional arguments with `repr`, then keyword arguments as `name=repr(value)`, separated by `", "`).

Both must pass every argument through and return the original's result.

In [ ]:
import functools

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_wrappers():\n    \"wrapping functions\"\n    def vat(pence, rate=20):\n        \"VAT on a price.\"\n        return pence * rate // 100\n    c = count_calls(vat)\n    assert c(1000) == 200 and c(500, rate=5) == 25 and c.calls == 2 and c.__name__ == \"vat\" and c.__doc__ == \"VAT on a price.\"\n    log = []\n    r = recorded(vat, log)\n    assert r(1000, rate=5) == 50 and r(999) == 199 and r.__name__ == \"vat\"\n    assert log == [\"vat(1000, rate=5) -> 50\", \"vat(999) -> 199\"], f\"log: {log}\"\n    greet = recorded(lambda name, punct=\"!\": f\"Hi {name}{punct}\", log)\n    greet(\"Ada\")\n    assert log[-1] == \"<lambda>('Ada') -> 'Hi Ada!'\"")

### Exercise: Retrying with settings

Write a callable class `Retry(func, times=3, errors=(ConnectionError,))`. Calling it calls `func` with the same arguments; if `func` raises one of the `errors`, it tries again, up to `times` attempts in all, then lets the last error through. Other errors pass straight through without retrying. It keeps `attempts`, the total number of calls made to `func` so far, and `failures`, a list of the error messages it caught (whether or not a later attempt succeeded). `times` below 1 raises `ValueError`.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_retry():\n    \"retrying\"\n    R = ns[\"Retry\"]\n    plan = iter([ConnectionError(\"timeout 1\"), ConnectionError(\"timeout 2\"), \"ok\"])\n    def flaky(x):\n        step = next(plan)\n        if isinstance(step, Exception):\n            raise step\n        return x * 2\n    r = R(flaky, times=3)\n    assert r(21) == 42 and r.attempts == 3 and r.failures == [\"timeout 1\", \"timeout 2\"]\n    def down():\n        raise ConnectionError(\"down\")\n    r2 = R(down, times=2)\n    try:\n        r2()\n        assert False, \"should give up after 2 attempts\"\n    except ConnectionError:\n        pass\n    assert r2.attempts == 2\n    def bad():\n        raise ValueError(\"bad input\")\n    r3 = R(bad)\n    try:\n        r3()\n        assert False\n    except ValueError:\n        pass\n    assert r3.attempts == 1 and r3.failures == []\n    assert R(lambda: 1, errors=(KeyError,))() == 1\n    try:\n        R(bad, times=0)\n        assert False, \"times 0 should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: Stackable writers

Writers have one method, `write(line)`. `ListWriter` (given) keeps every line. Write three decorators: `Upper(inner)` (passes the line in capitals), `Numbered(inner)` (adds `1: `, `2: `, and so on, counting its own lines), and `Censor(inner, words)` (replaces each of the words, matched in any case as whole words, with asterisks of the same length). Stack them in different orders and check what reaches the `ListWriter`.

In [ ]:
import re

class ListWriter:
    def __init__(self):
        self.lines = []

    def write(self, line):
        self.lines.append(line)

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_writers():\n    \"stacking\"\n    L, U, N, C = ns[\"ListWriter\"], ns[\"Upper\"], ns[\"Numbered\"], ns[\"Censor\"]\n    out = L()\n    w = Numbered(Censor(Upper(out), [\"darn\"]))\n    for line in [\"Darn it\", \"all good\", \"darned luck\"]:\n        w.write(line)\n    assert out.lines == [\"1: **** IT\", \"2: ALL GOOD\", \"3: DARNED LUCK\"], f\"got {out.lines}\"\n    out2 = L()\n    w2 = U(N(out2))\n    w2.write(\"hello\")\n    w2.write(\"bye\")\n    assert out2.lines == [\"1: HELLO\", \"2: BYE\"]\n    out3 = L()\n    C(L(), [\"x\"]).write(\"x\")\n    C(out3, [\"rats\", \"drat\"]).write(\"Rats! Drat, drats.\")\n    assert out3.lines == [\"****! ****, drats.\"]")

### Bug hunt: The wrapper that lost the result

After wrapping, `total` returns `None`, so this prints `None` instead of `1350` (after the log line). Fix the wrapper.

In [ ]:
import functools

def announce(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        func(*args, **kwargs)
    return wrapper

def total(prices):
    return sum(prices)

total = announce(total)
print(total([450, 900]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the result\"\n    expect_output(\"calling total\\n1350\")")

### Bug hunt: No keywords allowed

The timing wrapper only accepts positional arguments, so calling the wrapped function with a keyword crashes with `TypeError: wrapper() got an unexpected keyword argument 'rate'`. Fix the wrapper so it passes everything through, and the program prints `50` then `calls: 1`.

In [ ]:
import functools

def counted(func):
    @functools.wraps(func)
    def wrapper(*args):
        wrapper.calls += 1
        return func(*args)
    wrapper.calls = 0
    return wrapper

def vat(pence, rate=20):
    return pence * rate // 100

vat = counted(vat)
print(vat(1000, rate=5))
print("calls:", vat.calls)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keywords too\"\n    expect_output(\"50\\ncalls: 1\")")

### Bug hunt: Logging the wrong layer

The audit log should record only the lookups that actually reached the database, but it records every request, so this prints `['rice', 'rice', 'rice']` instead of `['rice']`. Fix the order of the decorators (not the classes).

In [ ]:
class Database:
    def get(self, key):
        return {"rice": 450}.get(key)

class Cached:
    def __init__(self, inner):
        self.inner, self.saved = inner, {}
    def get(self, key):
        if key not in self.saved:
            self.saved[key] = self.inner.get(key)
        return self.saved[key]

class Audited:
    def __init__(self, inner, log):
        self.inner, self.log = inner, log
    def get(self, key):
        self.log.append(key)
        return self.inner.get(key)

log = []
source = Audited(Cached(Database()), log)
for _ in range(3):
    source.get("rice")
print(log)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only real lookups\"\n    expect_output(\"['rice']\")")

### Puzzle: A function wrapper

Arrange the lines into a wrapper that prints how many items a function returned, keeping its name. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Base:
    def say(self, x):
        return x

class Twice:
    def __init__(self, inner):
        self.inner = inner
    def say(self, x):
        return self.inner.say(x) * 2

class Bang:
    def __init__(self, inner):
        self.inner = inner
    def say(self, x):
        return self.inner.say(x + "!")

p = ___   # Twice(Base()).say("hi")
q = ___   # Bang(Twice(Base())).say("hi")
r = ___   # Twice(Bang(Base())).say("hi")
s = ___   # Twice(Twice(Base())).say("a")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q) == (\"hihi\", \"hi!hi!\"), \"p and q\"\n    assert (r, s) == (\"hi!hi!\", \"aaaa\")")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a careful web client

Programs that fetch data from the internet usually need caching, limits on how often they ask, and a log. Build these as stackable decorators around anything with `fetch(url) -> str`, with the time passed in (Week 34) so everything can be tested:

- `FakeWeb(pages)`: a component for tests; `fetch(url)` returns `pages[url]`, or raises `LookupError` for an unknown URL; it counts its calls in `requests`.
- `Cached(inner, clock, seconds)`: remembers each URL's text for `seconds` seconds (by `clock()`, which returns a number of seconds); after that, it fetches again. Errors aren't cached.
- `Limited(inner, clock, per_minute)`: allows at most `per_minute` fetches in any 60-second window (like Week 31's limiter: a fetch at time `t` counts until `t + 60`); one more raises `RateLimited`, an exception you define, without calling the inner object.
- `Logged(inner, log)`: appends `"GET url"` before fetching and `"OK url"` or `"FAIL url: LookupError"` (the error's type name) after, and lets errors through.

Each decorator must work around any of the others, in any order.

In [ ]:
class RateLimited(Exception):
    pass

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parts():\n    \"each decorator\"\n    W, C, Li, Lo, RL = (ns[n] for n in [\"FakeWeb\", \"Cached\", \"Limited\", \"Logged\", \"RateLimited\"])\n    now = [0]\n    clock = lambda: now[0]\n    web = W({\"a\": \"page A\", \"b\": \"page B\"})\n    c = C(web, clock, 30)\n    assert c.fetch(\"a\") == \"page A\" and c.fetch(\"a\") == \"page A\" and web.requests == 1\n    now[0] = 31\n    assert c.fetch(\"a\") == \"page A\" and web.requests == 2\n    for _ in range(2):\n        try:\n            c.fetch(\"zzz\")\n            assert False\n        except LookupError:\n            pass\n    assert web.requests == 4, \"errors aren't cached\"\n    now[0] = 0\n    lim = Li(W({\"a\": \"A\"}), clock, 2)\n    lim.fetch(\"a\")\n    now[0] = 10\n    lim.fetch(\"a\")\n    try:\n        lim.fetch(\"a\")\n        assert False, \"the third fetch within 60 seconds should raise RateLimited\"\n    except RL:\n        pass\n    now[0] = 60\n    assert lim.fetch(\"a\") == \"A\", \"the fetch at time 0 no longer counts at time 60\"\n    log = []\n    lg = Lo(W({\"a\": \"A\"}), log)\n    lg.fetch(\"a\")\n    try:\n        lg.fetch(\"b\")\n    except LookupError:\n        pass\n    assert log == [\"GET a\", \"OK a\", \"GET b\", \"FAIL b: LookupError\"]\ndef test_stacking():\n    \"stacked\"\n    W, C, Li, Lo, RL = (ns[n] for n in [\"FakeWeb\", \"Cached\", \"Limited\", \"Logged\", \"RateLimited\"])\n    now = [0]\n    clock = lambda: now[0]\n    web = W({\"a\": \"A\", \"b\": \"B\"})\n    log = []\n    client = C(Lo(Li(web, clock, 2), log), clock, 100)\n    assert [client.fetch(u) for u in [\"a\", \"a\", \"b\", \"a\", \"b\"]] == [\"A\", \"A\", \"B\", \"A\", \"B\"]\n    assert web.requests == 2 and log == [\"GET a\", \"OK a\", \"GET b\", \"OK b\"]\n    other = Lo(Li(C(W({\"a\": \"A\"}), clock, 100), clock, 1), [])\n    other.fetch(\"a\")\n    try:\n        other.fetch(\"a\")\n        assert False, \"the limiter is outside the cache here, so the second request is refused\"\n    except RL:\n        pass\n    assert other.log[-1] == \"FAIL a: RateLimited\"")

## Warm-up

`source` is a `Prices` wrapped twice. Each layer has the same `get` method as the thing it wraps, does its own small job (recording, filling in a default) and passes the call inward. Callers of `source.get` can't tell how many layers there are. That's the Decorator pattern.

## Too many subclasses

Without decorators, adding optional behaviours by inheritance goes like this: `Prices`, then `LoggedPrices`, `CachedPrices`, `RetryingPrices`, then `LoggedCachedPrices`, `CachedRetryingPrices`, `LoggedCachedRetryingPrices`... Three optional behaviours make eight combinations, four make sixteen, and the same behaviours must be written again for every other kind of source (Week 30, Day 5 called this class explosion).

The Gang of Four's **Decorator** pattern says: "attach additional responsibilities to an object dynamically. Decorators provide a flexible alternative to subclassing for extending functionality." Each behaviour is written **once**, as a wrapper, and works with any object that has the right interface:

| Part | Role |
|---|---|
| component | the interface (here, anything with `get(item)`) |
| concrete component | a real thing doing the job (`Prices`) |
| decorator | holds a component, has the same interface, adds behaviour before or after passing the call on |

## Object decorators

Here are three reusable decorators for anything with `get(key)`:

- Each decorator does one job (Week 34's single responsibility) and knows nothing about the others.
- They're combined when the object is made, so different parts of a program can use different combinations of the same pieces.
- Because each has the same interface as what it wraps, a decorated object can go anywhere the original could (Liskov substitution).

If the interface has many methods, a decorator must pass every one of them on, even those it doesn't change. Week 31's `__getattr__` can forward the rest automatically, at the cost of being less explicit.

## The order of the layers

Decorators run from the outside in on the way to the component, and from the inside out on the way back. So the order of wrapping changes the result. Above, `Logged` sat **inside** `Cached`, so it only saw the calls that reached the database. Swap them:

Neither order is wrong: logging every request and logging every database query are both useful. The point is to decide on purpose. A good habit is to read a stack of decorators from the outside in, saying what each layer does to the call before passing it on.

## Wrapping functions

Functions can be decorated too: a function that takes a function and returns a new one that adds behaviour and calls the original. Week 31's `Remember` did this with a class; with a closure (Week 7) it looks like this:

The parts of every function wrapper:

- `wrapper(*args, **kwargs)` accepts **any** arguments and passes them all on, so it works for any function (Week 7's `*args` and `**kwargs`);
- it calls the original, does its extra job before or after, and **returns the original's result**;
- the outer function returns `wrapper` itself (not `wrapper()`);
- `@functools.wraps(func)` copies the original's name, docstring and other details onto `wrapper`, so `vat.__name__` is still `"vat"`, not `"wrapper"`. That `@` line is Python's decorator syntax, used here on `wrapper`; Phase 5 explains how it works.

`vat = logged(vat)` replaces the name with the wrapped version, as `ways = Remember(ways)` did in Week 31. Writing `@logged` on the line above `def vat` does exactly the same thing, which is why that syntax is called a "decorator": it's this pattern, built into the language. `@functools.cache`, which you've used since Week 16, is a function decorator from the standard library.

When a decorator needs **settings** (how many times to retry, say), a callable class (Week 31) is a clear way to write it:

## Decorator, Adapter, Proxy

Three patterns wrap an object, for different reasons:

| Pattern | Interface | Purpose |
|---|---|---|
| Decorator | the **same** as the wrapped object | add behaviour (logging, caching, retrying) |
| Adapter (Day 4) | **different**: the one the client expects | make something fit |
| Proxy | the **same** | control access to the real object: create it only when first needed, check permissions, or stand in for something far away |

A caching wrapper is sometimes called a proxy and sometimes a decorator; the names describe intent, and the code can look identical. What matters is knowing why the wrapper is there.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a function in your own code that you'd like to time or log without changing it, and wrap it with today's `logged` (as `name = logged(name)`). Tomorrow is the Forge: all five patterns working together in one program.